# 03. Térbeli Elemzés és Interaktív GIS Térképek

**Cél:** az ingatlanárak makrotérbeli eloszlásának, a lokációs gócpontoknak és az urbánus térszerkezetnek a vizualizációja a garantált pontosságú almintán.

---

### 📖 A lokáció elsődlegessége
Az ingatlanárazás elsődleges tényezője nem az épület, hanem az általa elfoglalt tér értéke. Az urbánus tér nem izotróp: a központok, közlekedési tengelyek és zöldterületek körül „érték-hegyek", az ipari, barnamezős vagy zajterhelt zónák körül „érték-völgyek" alakulnak ki. A GIS-vizualizációk ezeket az összefüggéseket teszik láthatóvá.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: {len(df_pontos)} db.")

### 1. Ponttérkép és Fajlagos Árak Térbeli Eloszlása

**📌 Interpretációs útmutató:**
A térképen minden pont egy ingatlant jelöl, a színskála a fajlagos árat kódolja. A hasonló árú ingatlanok csoportosulása vizuálisan is jelzi a **térbeli autokorreláció** jelenlétét, amelyet a 08. fejezet statisztikailag is tesztel (Tobler első földrajzi törvénye).

In [ ]:
n_pontos = len(df_pontos)
in_vasut_10p = int((df_pontos['vasut_10p_seta'] == 1).sum()) if 'vasut_10p_seta' in df_pontos.columns else 0
in_metro_10p = int((df_pontos['metro_10p_seta'] == 1).sum()) if 'metro_10p_seta' in df_pontos.columns else 0
poi_15_avg = float(df_pontos['poi_15p_count'].mean()) if 'poi_15p_count' in df_pontos.columns else 0.0
med_nm_ar_pontos = df_pontos['nm_ar_huf'].median()

kpi_cards = [
    ("Garantált Pontos Minta", f"{n_pontos} db", "100% valós koordináta (eladó)", "#1e3a8a"),
    ("Vasútállomás 10p Séta", f"{in_vasut_10p} db", f"{in_vasut_10p/n_pontos*100:.1f}% lefedettség", "#ec4899"),
    ("Metró 10p Séta", f"{in_metro_10p} db", f"{in_metro_10p/n_pontos*100:.1f}% lefedettség", "#2563eb"),
    ("POI 15p Átlag", f"{poi_15_avg:.1f} db", "Hálózati 15-perces ellátottság", "#059669"),
    ("Pontos Minta Medián Ár", fmt_huf(med_nm_ar_pontos), "Fajlagos eladási ár/m²", "#7c3aed"),
    ("Városrészek Száma", f"{df_pontos['varosresz'].nunique()} db", "Térbeli lefedettség", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Fajlagos Árak Rácstérképe: Cellánkénti Medián és IDW-Interpolált Árfelület

**📌 A két ábra logikája:**
A korábbi px.density_map hőtérkép gyakorlatilag csak magukat a pontokat mutatta, ezért két, ténylegesen informatív vizualizációra cseréltük:
- **Cellánkénti medián térkép (~150 m-es rács, N≥3):** a garantált pontosságú pontokat lokális síkvetületre vetítjük (x = lon·cos(mid_lat)·111 320 m, y = lat·110 540 m), ~150 m-es négyzetrácsra binezzük — ez a hexagonális binezés dokumentált, egyszerűbb közelítése —, és cellánként a fajlagos ár mediánját rajzoljuk a cellaközéppontra. A pontméret a cella elemszámát kódolja; az N≥3 szűrés az egypontos cellák zaját szűri ki.
- **IDW-interpolált árfelület (power=2):** ugyanazon a vetületen ~75 m-es rácsra inverz távolság-súlyozással (IDW, p=2) folytonos árfelületet simítunk kizárólag a pontos pontokból, és kontúrtérképként (go.Contour) ábrázoljuk.

**🔍 Városfejlesztési konklúzió:**
A két térkép gradiensvonalai a városszövet fizikai törésvonalait (vágányhálózatok, főutak, ipari zónák) rajzolják ki, és kijelölik az „érték-hegyeket" (prémium zónák), valamint az „érték-völgyeket" (infrastrukturális deficittel vagy környezeti terheléssel küzdő zónák).


In [ ]:
fig1 = px.scatter_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='nm_ar_huf',
    size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True, 'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[df_pontos['nm_ar_huf'].quantile(0.05), df_pontos['nm_ar_huf'].quantile(0.95)],
    zoom=12.2,
    center=get_map_center(),
    map_style='carto-positron',
    title='Lakások térbeli elhelyezkedése és négyzetméterára (Carto Positron térképen)'
)
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

### 3. Térbeli Árprofil (Kelet-Nyugat és Észak-Dél)

**📌 Mit mutat a keresztmetszeti ábra?**
A területen átívelő ártrendeket:
- **Kelet-Nyugati irány**: A belső városmagtól a periféria felé haladva a négyzetméterárak fokozatosan mérséklődnek.
- **Észak-Déli irány**: Északon jellemzően magasabb az árszint, mint a déli ipari zónák felé haladva.

In [ ]:
# ------------------------------------------------------------------
# 2A. Cellánkénti medián négyzetméterár térkép (~150 m-es rács, N≥3)
# A korábbi px.density_map hőtérkép helyett: saját rácsozás lokális síkvetületen.
# Vetítés: x = lon*cos(mid_lat)*111320, y = lat*110540 (lokális méter).
# A ~150 m-es NÉGYZETrács a hexagonális binezés dokumentált közelítése.
df_ar = df_pontos.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
mid_lat = float(df_ar['geokodolt_lat'].mean())
lon_scale = 111320.0 * np.cos(np.radians(mid_lat))
lat_scale = 110540.0
df_ar['x_proj'] = df_ar['geokodolt_lon'] * lon_scale
df_ar['y_proj'] = df_ar['geokodolt_lat'] * lat_scale

GRID_M = 150.0  # ~150 m-es cellaméret
x0 = float(df_ar['x_proj'].min())
y0 = float(df_ar['y_proj'].min())
df_ar['cell_i'] = np.floor((df_ar['x_proj'] - x0) / GRID_M).astype(int)
df_ar['cell_j'] = np.floor((df_ar['y_proj'] - y0) / GRID_M).astype(int)

cell_stats = (
    df_ar.groupby(['cell_i', 'cell_j'])['nm_ar_huf']
    .agg(['count', 'median'])
    .rename(columns={'count': 'N', 'median': 'median_ar'})
    .reset_index()
)
cell_stats = cell_stats[cell_stats['N'] >= 3].copy()  # N≥3: egypontos cellák zajszűrése
cell_stats['x_cell'] = x0 + (cell_stats['cell_i'] + 0.5) * GRID_M
cell_stats['y_cell'] = y0 + (cell_stats['cell_j'] + 0.5) * GRID_M
cell_stats['cell_lat'] = cell_stats['y_cell'] / lat_scale
cell_stats['cell_lon'] = cell_stats['x_cell'] / lon_scale

if len(cell_stats) == 0:
    print('FIGYELEM: nincs N≥3 elemszámú cella — a medián-térkép üres.')
    fig2 = go.Figure()
    fig2.add_annotation(text='Nincs elegendő adat az N≥3 rácstérképhez', showarrow=False)
    fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
else:
    _ar_lo = float(cell_stats['median_ar'].quantile(0.05))
    _ar_hi = float(cell_stats['median_ar'].quantile(0.95))
    # Statikus HTML-exportban a scatter_map (token-mentes) a megbízható renderer;
    # ha van MAPBOX_TOKEN, a px.scatter_mapbox is használható.
    _mapbox_token = os.environ.get('MAPBOX_TOKEN')
    if _mapbox_token:
        px.set_mapbox_access_token(_mapbox_token)
        fig2 = px.scatter_mapbox(
            cell_stats, lat='cell_lat', lon='cell_lon',
            size='N', color='median_ar',
            hover_data={'median_ar': ':.0f', 'N': True},
            size_max=25,
            color_continuous_scale='Viridis', range_color=[_ar_lo, _ar_hi],
            zoom=12.2, center=get_map_center(), mapbox_style='carto-positron',
            title='Cellánkénti medián négyzetméterár (rács ~150 m, N≥3)'
        )
    else:
        fig2 = px.scatter_map(
            cell_stats, lat='cell_lat', lon='cell_lon',
            size='N', color='median_ar',
            hover_data={'median_ar': ':.0f', 'N': True},
            size_max=25,
            color_continuous_scale='Viridis', range_color=[_ar_lo, _ar_hi],
            zoom=12.2, center=get_map_center(), map_style='carto-positron',
            title='Cellánkénti medián négyzetméterár (rács ~150 m, N≥3)'
        )
    fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
    print(f"Mediánár-rács: {len(cell_stats)} cella (N≥3, ~150 m), "
          f"cellánkénti medián tartomány: {cell_stats['median_ar'].min():,.0f} – {cell_stats['median_ar'].max():,.0f} Ft/m²")
fig2.show()

# ------------------------------------------------------------------
# 2B. IDW-interpolált árfelület (go.Contour) — ~75 m-es rács, power=2
IDW_GRID_M = 75.0
_pad = 150.0  # enyhe széltágítás, hogy a domén szélén is értelmezhető maradjon a felület
gx = np.arange(float(df_ar['x_proj'].min()) - _pad,
               float(df_ar['x_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
gy = np.arange(float(df_ar['y_proj'].min()) - _pad,
               float(df_ar['y_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
GX, GY = np.meshgrid(gx, gy)
grid_nodes = np.column_stack((GX.ravel(), GY.ravel()))
pts = df_ar[['x_proj', 'y_proj']].to_numpy(dtype=float)
vals = df_ar['nm_ar_huf'].to_numpy(dtype=float)

# IDW (power=2) minden rácspontra, kizárólag a pontos pontokból
z_flat = np.full(len(grid_nodes), np.nan)
for i in range(0, len(grid_nodes), 512):  # memóriabarát darabolás
    d = grid_nodes[i:i + 512, None, :] - pts[None, :, :]
    dist2 = np.einsum('ijk,ijk->ij', d, d)
    w = 1.0 / np.maximum(dist2, 1e-9)  # p=2; az eps a pontra eső rácspontokat védi
    z_flat[i:i + 512] = (w @ vals) / w.sum(axis=1)
Z = z_flat.reshape(GX.shape)

_z_lo, _z_hi = np.nanpercentile(Z, [2, 98])
fig2_idw = go.Figure(go.Contour(
    z=Z, x=gx, y=gy,
    colorscale='Viridis',
    contours=dict(coloring='heatmap', showlabels=False),
    zmin=_z_lo, zmax=_z_hi,
    colorbar=dict(title='Ft/m²'),
    hovertemplate='Kelet–Nyugat: %{x:.0f} m<br>Észak–Dél: %{y:.0f} m<br>Interpolált ár: %{z:,.0f} Ft/m²<extra></extra>'
))
fig2_idw.update_layout(
    title='Interpolált árfelület (IDW, power=2)',
    xaxis_title='Kelet–Nyugat (m, lokális síkvetület)',
    yaxis_title='Észak–Dél (m, lokális síkvetület)',
    height=520, template=PLOTLY_TEMPLATE
)
fig2_idw.update_yaxes(scaleanchor='x', scaleratio=1.0)
fig2_idw.show()
print(f"IDW-rács: {len(gx)}×{len(gy)} = {len(grid_nodes):,} rácspont (~75 m), "
      f"interpolált tartomány (2–98%): {_z_lo:,.0f} – {_z_hi:,.0f} Ft/m²")

# Vasúti vágánytól mért légvonali távolság vs Ár/m² (immissziós sávok szerint színezve)
fig3 = px.scatter(
    df_pontos,
    x='tavolsag_vasut_m',
    y='nm_ar_huf',
    color='vasut_zona',
    trendline='lowess',
    trendline_options=dict(frac=0.5),
    title='Vasúti vágánytól mért légvonali távolság (m) vs. Négyzetméterár (LOWESS, frac=0.5)',
    labels={'tavolsag_vasut_m': 'Vágánytól mért légvonali távolság (méter)', 'nm_ar_huf': 'Ár / m² (HUF)', 'vasut_zona': 'Immissziós sáv'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=480)
fig3.show()


### 4. Interaktív Térképi Szűrő Pult

**📌 Mire használható a szűrő?**
Árkategória és alapterület szerint fókuszálhat a térképen lévő ingatlanokra, megfigyelve a különböző kategóriák térbeli tömörülését.

In [ ]:
# Interaktív Többváltozós Térbeli Térkép (Plotly updatemenus)
map_sub = df[df['minta_garantalt_pontos'] == 1].copy()

map_vars = [
    ('nm_ar_huf', '1. Fajlagos Négyzetméterár (HUF/m²)', 'Viridis', ':.0f'),
    ('ar_millio_ft', '2. Kínálati Vételár (M Ft)', 'Plasma', ':.1f'),
    ('alapterulet_nm', '3. Lakás Alapterület (m²)', 'Blues', ':.0f'),
    ('szobaszam_osszes', '4. Szobaszám (db)', 'Purples', ':.1f'),
    ('allapot_kod', '5. Műszaki Állapot Kód (1-6)', 'Greens', ':.0f'),
    ('tavolsag_metro_halozati_m', '6. Metróállomás Hálózati Sétaút (m)', 'RdYlBu', ':.0f'),
    ('tavolsag_vasut_m', '7. Vasúti Pálya Légvonal / Zaj (m)', 'Spectral', ':.0f')
]

fig_map = go.Figure()
buttons = []

for i, (col, label, colscale, fmt) in enumerate(map_vars):
    sub_f = px.scatter_map(
        map_sub,
        lat='geokodolt_lat',
        lon='geokodolt_lon',
        color=col,
        size='alapterulet_nm',
        size_max=12,
        hover_name='cim_teljes',
        hover_data={'varosresz': True, col: fmt, 'alapterulet_nm': ':.0f'},
        color_continuous_scale=colscale,
        zoom=12.2,
        center=get_map_center(),
        map_style='carto-positron'
    )
    tr = sub_f.data[0]
    tr.visible = (i == 0)
    tr.name = label
    fig_map.add_trace(tr)
    
    vis = [j == i for j in range(len(map_vars))]
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis}, {'title': f'Interaktív Térbeli Eloszlás: {label} (N={len(map_sub)})'}]
    ))

fig_map.update_layout(
    title=f'Interaktív Térbeli Eloszlás: {map_vars[0][1]} (N={len(map_sub)})',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    map_style='carto-positron',
    map_zoom=12.2,
    map_center=get_map_center(),
    height=580,
    margin={"r":0,"t":50,"l":0,"b":0}
)
fig_map.show()